In [6]:
import os
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from typing import Annotated, List, Optional, Dict, Literal

load_dotenv()

True

In [8]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [1]:
# pip install langgraph-supervisor

In [16]:
@tool
def python_repl_tool(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Execute Python code in a controlled REPL and return stdout."""
    try:
        result = repl.run(code)
    except Exception as e:
        return f"Failed to execute.\nError: {type(e).__name__}: {e}"

    stdout = result if result is not None else "(no stdout)"
    return (
        f"Successfully executed:\n```python\n{code}\n```\n"
        f"Stdout: {stdout}\n\n"
        "If you have completed all tasks, respond with FINAL ANSWER."
    )


Setup a search tool

In [9]:
search_tool=TavilySearchResults(tavily_api_key=TAVILY_API_KEY)

C:\Users\DELL\AppData\Local\Temp\ipykernel_9708\549769495.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool=TavilySearchResults(tavily_api_key=TAVILY_API_KEY)


Initialise LLM

In [10]:
llm = ChatOpenAI(model="gpt-4o")

Creating worker agents

Reaserach Agent

In [11]:
research_agent = create_react_agent(
    model=llm, 
    name="research_agent",
    tools=[search_tool], 
    prompt="You can only do research. You do not generate charts")

C:\Users\DELL\AppData\Local\Temp\ipykernel_9708\2632632928.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(


Code Agent

In [12]:
code_agent=create_react_agent(
    model=llm,
    name="code_agent",
    tools=[python_repl_tool],
    prompt="You can generate charts . DO NOT do any research.")

C:\Users\DELL\AppData\Local\Temp\ipykernel_9708\3502500287.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  code_agent=create_react_agent(


Create a Supervisor

In [13]:
workflow = create_supervisor(
    [research_agent, code_agent],
    model=llm,
    prompt=(
        "You are a team supervisor managing a research expert and a math expert. "
        "For research, use research_agent. "
        "For creating charts, use code_agent."
    )
)

In [17]:
app = workflow.compile()
app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what's the GDP of UK in past three years, draw a line chart?"
        }
    ]
})

{'messages': [HumanMessage(content="what's the GDP of UK in past three years, draw a line chart?", additional_kwargs={}, response_metadata={}, id='acd54e95-dd92-41ec-abfc-6afba1b9003e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 107, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c7d0c94d9d', 'id': 'chatcmpl-DEWMgTtEHHobmw2KD86AkWQQiMoUl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019ca87e-4940-7ff2-b4af-ed13c4fed4e5-0', tool_calls=[{'name': 'transfer_to_research_agent', 'args': {}, 'id': 'call_etzQgh9Vu79wSg5gELcD0mvk', 'type': 'tool_call'}], invalid_tool_calls=